In [1]:
import numpy as np
import math
import random
import csv
import pandas as pd
import scipy
import scipy.optimize as opt
from bayes_opt import BayesianOptimization
from bayes_opt.logger import JSONLogger
from bayes_opt.event import Events

In [25]:
def P_action_softmax(value,gamma,state,action):
    Q = value[:,0]
    exp_Q = np.sum(np.exp(gamma*Q))
    return (np.exp(gamma*Q)/exp_Q)[action]

def P_action_con1(value,gamma):
    Q = value[:,1]
    exp_Q = np.sum(np.exp(Q*gamma))
    return np.exp(Q*gamma)/exp_Q

def P_action_con2(value,gamma):
    Q = value[:,2]
    exp_Q = np.sum(np.exp(Q*gamma))
    return np.exp(Q*gamma)/exp_Q
    
def P_action_con0(value,gamma):
    Q1 = (value[0,1]+value[0,2])/2
    Q2 = (value[1,1]+value[1,2])/2
    Q = np.array([Q1,Q2])
    exp_Q = np.sum(np.exp(Q*gamma))
    return np.exp(Q*gamma)/exp_Q

def learn(value,lr,states,actions,reward):
    if states==[0,1,0] or states==[0,0,1]:
        value[1,0] = value[1,0] + lr * (reward - 1 - value[1,0])
    else :
        value[0,0] = value[0,0] + lr * (reward - value[0,0])
    if actions==1:
        if states==[0,1,0]:
            value[1,1] = value[1,1] + lr*(reward-value[1,1])      
        elif states==[0,0,1]:
            value[1,2] = value[1,2] + lr*(reward-value[1,2])    
        else:
            value[1,1] = value[1,1] + 0.5*lr*(reward-value[1,1])      
            value[1,2] = value[1,2] + 0.5*lr*(reward-value[1,2])
    return value

def read_behavioral_data(n):
    result_stay_cue = []
    result_safe_risk = []
    action_stay_cue = []
    action_safe_risk = []
    if_can_ask = []
    fname = './behavioral_data/uncertainty_' + str(n+1) + '_2022.csv'
    with open(fname,'r') as f :
        for line in f.readlines():
            if line.split(',')[0]=='0' or line.split(',')[0]=='1':
                if int(line.split(',')[3])==0:
                    action_stay_cue.append(0)
                elif int(line.split(',')[3])==1 or int(line.split(',')[3])==2:
                    action_stay_cue.append(1)
                else :
                    print('ERROR')
                result_stay_cue.append(int(line.split(',')[3]))
                result_safe_risk.append(int(float(line.split(',')[6])))
                action_safe_risk.append(int(line.split(',')[4]))
                if line.split(',')[1]==' ':
                    if_can_ask.append(0)
                else :
                    if_can_ask.append(1)

    return if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk

def P_model_free_rl(gamma,lr):
    subject = 0
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

In [17]:
# def P_action_stay_cue(value,gamma,state,action):
#     Q = value[:,0]
#     exp_Q = np.sum(np.exp(gamma*Q))
#     return (np.exp(gamma*Q)/exp_Q)[action]

# def P_action_safe_risk(value,gamma):
#     Q = value[:,1]
#     exp_Q = np.sum(np.exp(Q*gamma))
#     return np.exp(Q*gamma)/exp_Q

# def learn(value,lr,states,actions,reward):
#     if states==[0,1,0] or states==[0,0,1]:
#         value[1,0] = value[1,0] + lr * (reward - 1 - value[1,0])
#     else :
#         value[0,0] = value[0,0] + lr * (reward - value[0,0])
#     if actions==1:
#         value[1,1] = value[1,1] + lr*(reward-value[1,1])      
#     return value

# def read_behavioral_data(n):
#     result_stay_cue = []
#     result_safe_risk = []
#     action_stay_cue = []
#     action_safe_risk = []
#     if_can_ask = []
#     fname = './behavioral_data/uncertainty_' + str(n+1) + '_2022.csv'
#     with open(fname,'r') as f :
#         for line in f.readlines():
#             if line.split(',')[0]=='0' or line.split(',')[0]=='1':
#                 if int(line.split(',')[3])==0:
#                     action_stay_cue.append(0)
#                 elif int(line.split(',')[3])==1 or int(line.split(',')[3])==2:
#                     action_stay_cue.append(1)
#                 else :
#                     print('ERROR')
#                 result_stay_cue.append(int(line.split(',')[3]))
#                 result_safe_risk.append(int(float(line.split(',')[6])))
#                 action_safe_risk.append(int(line.split(',')[4]))
#                 if line.split(',')[1]==' ':
#                     if_can_ask.append(0)
#                 else :
#                     if_can_ask.append(1)

#     return if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk

# def P_model_free_rl(gamma,lr):
#     subject = 0
#     if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
#     log_p = 0
#     value = np.array([[6,6],[6,6]])
#     for i in range(120):
#         if if_can_ask[i] == 1:
#             log_p += np.log(P_action_stay_cue(value,gamma,[1,0,0],action_stay_cue[i]))
#         if result_stay_cue[i]==0:
#             state = [0,0.5,0.5]
#             log_p += np.log(P_action_safe_risk(value,gamma)[action_safe_risk[i]])
#         elif result_stay_cue[i]==1:
#             state = [0,1,0]
#             log_p += np.log(P_action_safe_risk(value,gamma)[action_safe_risk[i]])
#         elif result_stay_cue[i]==2:
#             state=[0,0,1]
#             log_p += np.log(P_action_safe_risk(value,gamma)[action_safe_risk[i]])
#         else:
#             print('ERROR')
#         value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
#     return log_p

In [29]:
def P_model_free_rl_1(gamma,lr):
    subject = 0
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_1 = BayesianOptimization(
    f=P_model_free_rl_1,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_1 = JSONLogger(path="./logs_rl_1.log")
optimizer_rl_1.subscribe(Events.OPTIMIZATION_STEP, logger_rl_1)
optimizer_rl_1.maximize(
    init_points=1000,
    n_iter=1000,
)

In [30]:
def P_model_free_rl_2(gamma,lr):
    subject = 1
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_2 = BayesianOptimization(
    f=P_model_free_rl_2,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_2 = JSONLogger(path="./logs_rl_2.log")
optimizer_rl_2.subscribe(Events.OPTIMIZATION_STEP, logger_rl_2)
optimizer_rl_2.maximize(
    init_points=1000,
    n_iter=1000,
)

In [31]:
def P_model_free_rl_3(gamma,lr):
    subject = 2
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_3 = BayesianOptimization(
    f=P_model_free_rl_3,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_3 = JSONLogger(path="./logs_rl_3.log")
optimizer_rl_3.subscribe(Events.OPTIMIZATION_STEP, logger_rl_3)
optimizer_rl_3.maximize(
    init_points=1000,
    n_iter=1000,
)

In [32]:
def P_model_free_rl_4(gamma,lr):
    subject = 3
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_4 = BayesianOptimization(
    f=P_model_free_rl_4,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_4 = JSONLogger(path="./logs_rl_4.log")
optimizer_rl_4.subscribe(Events.OPTIMIZATION_STEP, logger_rl_4)
optimizer_rl_4.maximize(
    init_points=1000,
    n_iter=1000,
)

In [33]:
def P_model_free_rl_5(gamma,lr):
    subject = 4
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_5 = BayesianOptimization(
    f=P_model_free_rl_5,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_5 = JSONLogger(path="./logs_rl_5.log")
optimizer_rl_5.subscribe(Events.OPTIMIZATION_STEP, logger_rl_5)
optimizer_rl_5.maximize(
    init_points=1000,
    n_iter=1000,
)

In [34]:
def P_model_free_rl_6(gamma,lr):
    subject = 5
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_6 = BayesianOptimization(
    f=P_model_free_rl_6,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_6 = JSONLogger(path="./logs_rl_6.log")
optimizer_rl_6.subscribe(Events.OPTIMIZATION_STEP, logger_rl_6)
optimizer_rl_6.maximize(
    init_points=1000,
    n_iter=1000,
)

In [35]:
def P_model_free_rl_7(gamma,lr):
    subject = 6
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_7 = BayesianOptimization(
    f=P_model_free_rl_7,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_7 = JSONLogger(path="./logs_rl_7.log")
optimizer_rl_7.subscribe(Events.OPTIMIZATION_STEP, logger_rl_7)
optimizer_rl_7.maximize(
    init_points=1000,
    n_iter=1000,
)

In [36]:
def P_model_free_rl_8(gamma,lr):
    subject = 7
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_8 = BayesianOptimization(
    f=P_model_free_rl_8,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_8 = JSONLogger(path="./logs_rl_8.log")
optimizer_rl_8.subscribe(Events.OPTIMIZATION_STEP, logger_rl_8)
optimizer_rl_8.maximize(
    init_points=1000,
    n_iter=1000,
)

In [37]:
def P_model_free_rl_9(gamma,lr):
    subject = 8
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_9 = BayesianOptimization(
    f=P_model_free_rl_9,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_9 = JSONLogger(path="./logs_rl_9.log")
optimizer_rl_9.subscribe(Events.OPTIMIZATION_STEP, logger_rl_9)
optimizer_rl_9.maximize(
    init_points=1000,
    n_iter=1000,
)

In [38]:
def P_model_free_rl_10(gamma,lr):
    subject = 9
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_10 = BayesianOptimization(
    f=P_model_free_rl_10,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_10 = JSONLogger(path="./logs_rl_10.log")
optimizer_rl_10.subscribe(Events.OPTIMIZATION_STEP, logger_rl_10)
optimizer_rl_10.maximize(
    init_points=1000,
    n_iter=1000,
)

KeyboardInterrupt: 

In [ ]:
def P_model_free_rl_11(gamma,lr):
    subject = 10
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_11 = BayesianOptimization(
    f=P_model_free_rl_11,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_11 = JSONLogger(path="./logs_rl_11.log")
optimizer_rl_11.subscribe(Events.OPTIMIZATION_STEP, logger_rl_11)
optimizer_rl_11.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_free_rl_12(gamma,lr):
    subject = 11
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_12 = BayesianOptimization(
    f=P_model_free_rl_12,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_12 = JSONLogger(path="./logs_rl_12.log")
optimizer_rl_12.subscribe(Events.OPTIMIZATION_STEP, logger_rl_12)
optimizer_rl_12.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_free_rl_13(gamma,lr):
    subject = 12
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_13 = BayesianOptimization(
    f=P_model_free_rl_13,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_13 = JSONLogger(path="./logs_rl_13.log")
optimizer_rl_13.subscribe(Events.OPTIMIZATION_STEP, logger_rl_13)
optimizer_rl_13.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_free_rl_14(gamma,lr):
    subject = 13
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_14 = BayesianOptimization(
    f=P_model_free_rl_14,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_14 = JSONLogger(path="./logs_rl_14.log")
optimizer_rl_14.subscribe(Events.OPTIMIZATION_STEP, logger_rl_14)
optimizer_rl_14.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_free_rl_15(gamma,lr):
    subject = 14
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_15 = BayesianOptimization(
    f=P_model_free_rl_15,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_15 = JSONLogger(path="./logs_rl_15.log")
optimizer_rl_15.subscribe(Events.OPTIMIZATION_STEP, logger_rl_15)
optimizer_rl_15.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_free_rl_16(gamma,lr):
    subject = 15
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_16 = BayesianOptimization(
    f=P_model_free_rl_16,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_16 = JSONLogger(path="./logs_rl_16.log")
optimizer_rl_16.subscribe(Events.OPTIMIZATION_STEP, logger_rl_16)
optimizer_rl_16.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_free_rl_17(gamma,lr):
    subject = 16
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_17 = BayesianOptimization(
    f=P_model_free_rl_17,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_17 = JSONLogger(path="./logs_rl_17.log")
optimizer_rl_17.subscribe(Events.OPTIMIZATION_STEP, logger_rl_17)
optimizer_rl_17.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_free_rl_18(gamma,lr):
    subject = 17
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_18 = BayesianOptimization(
    f=P_model_free_rl_18,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_18 = JSONLogger(path="./logs_rl_18.log")
optimizer_rl_18.subscribe(Events.OPTIMIZATION_STEP, logger_rl_18)
optimizer_rl_18.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_free_rl_19(gamma,lr):
    subject = 18
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_19 = BayesianOptimization(
    f=P_model_free_rl_19,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_19 = JSONLogger(path="./logs_rl_19.log")
optimizer_rl_19.subscribe(Events.OPTIMIZATION_STEP, logger_rl_19)
optimizer_rl_19.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_free_rl_20(gamma,lr):
    subject = 19
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_20 = BayesianOptimization(
    f=P_model_free_rl_20,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_20 = JSONLogger(path="./logs_rl_20.log")
optimizer_rl_20.subscribe(Events.OPTIMIZATION_STEP, logger_rl_20)
optimizer_rl_20.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_free_rl_21(gamma,lr):
    subject = 20
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_21 = BayesianOptimization(
    f=P_model_free_rl_21,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_21 = JSONLogger(path="./logs_rl_21.log")
optimizer_rl_21.subscribe(Events.OPTIMIZATION_STEP, logger_rl_21)
optimizer_rl_21.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_free_rl_22(gamma,lr):
    subject = 21
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_22 = BayesianOptimization(
    f=P_model_free_rl_22,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_22 = JSONLogger(path="./logs_rl_22.log")
optimizer_rl_22.subscribe(Events.OPTIMIZATION_STEP, logger_rl_22)
optimizer_rl_22.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_free_rl_23(gamma,lr):
    subject = 22
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_23 = BayesianOptimization(
    f=P_model_free_rl_23,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_23 = JSONLogger(path="./logs_rl_23.log")
optimizer_rl_23.subscribe(Events.OPTIMIZATION_STEP, logger_rl_23)
optimizer_rl_23.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_free_rl_24(gamma,lr):
    subject = 23
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_24 = BayesianOptimization(
    f=P_model_free_rl_24,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_24 = JSONLogger(path="./logs_rl_24.log")
optimizer_rl_24.subscribe(Events.OPTIMIZATION_STEP, logger_rl_24)
optimizer_rl_24.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_free_rl_25(gamma,lr):
    subject = 24
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl_25 = BayesianOptimization(
    f=P_model_free_rl_25,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)
logger_rl_25 = JSONLogger(path="./logs_rl_25.log")
optimizer_rl_25.subscribe(Events.OPTIMIZATION_STEP, logger_rl_25)
optimizer_rl_25.maximize(
    init_points=1000,
    n_iter=1000,
)